# Training a CNN on MNIST with PyTorch Lightning

This notebook demonstrates how to train a Convolutional Neural Network (CNN) on the MNIST dataset using PyTorch Lightning.

In [ ]:
!pip uninstall rich -y

In [ ]:
!pip install pytorch-lightning torch torchvision datasets wandb pillow scikit-learn

In [ ]:
import torch
torch.set_float32_matmul_precision('high')  # or 'medium'

In [ ]:
# Install dependencies (uncomment if running in a fresh environment)
# !pip install pytorch-lightning torch torchvision datasets wandb

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import transforms
import pytorch_lightning as pl
import os
from datasets import load_dataset
from pytorch_lightning.loggers import WandbLogger

In [ ]:
# (Optional) Login to Weights & Biases if running interactively
import wandb
wandb.login()

In [ ]:
from torchvision.datasets import MNIST

class MNISTDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=1024, num_workers=2):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers
        # Enhanced data augmentation for training
        self.train_transform = transforms.Compose([
            transforms.RandomResizedCrop(28, scale=(0.8, 1.2), ratio=(0.9, 1.1)),
            transforms.RandomRotation(degrees=20),
            transforms.RandomAffine(degrees=15, translate=(0.15, 0.15), scale=(0.85, 1.15), shear=10),
            transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.3), value='random')
        ])
        # No augmentation for validation/test
        self.test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

    def prepare_data(self):
        MNIST(root="./data", train=True, download=True)
        MNIST(root="./data", train=False, download=True)

    def setup(self, stage=None):
        mnist_full = MNIST(root="./data", train=True, transform=self.train_transform)
        self.mnist_train, self.mnist_val = torch.utils.data.random_split(
            mnist_full, [55000, 5000], generator=torch.Generator().manual_seed(42)
        )
        # For validation, override transform to no augmentation
        self.mnist_val.dataset.transform = self.test_transform
        self.mnist_test = MNIST(root="./data", train=False, transform=self.test_transform)

    def train_dataloader(self):
        return DataLoader(self.mnist_train, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.mnist_val, batch_size=self.batch_size, num_workers=self.num_workers, pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self.mnist_test, batch_size=self.batch_size, num_workers=self.num_workers, pin_memory=True)

import torch.nn.init as init

class LitCNN(pl.LightningModule):
    def __init__(self, lr=1e-3, use_compile=False, compile_mode=None):
        super().__init__()
        self.save_hyperparameters()
        self.model = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, 1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, 1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(128, 256, 3, 1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Flatten(),
            nn.Linear(256 * 7 * 7, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )
        self.criterion = nn.CrossEntropyLoss()
        self.apply(self._init_weights)
        # For confusion matrix logging
        self.validation_step_outputs = []
        # Torch compile support
        self.use_compile = use_compile
        self.compile_mode = compile_mode
        self._compiled = False

    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d):
            init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                init.constant_(m.bias, 0)
        elif isinstance(m, nn.Linear):
            init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm2d) or isinstance(m, nn.BatchNorm1d):
            init.constant_(m.weight, 1)
            init.constant_(m.bias, 0)

    def forward(self, x):
        # Compile model on first forward if requested
        if self.use_compile and not self._compiled:
            try:
                self.model = torch.compile(self.model, mode=self.compile_mode)
                self._compiled = True
                print(f"Model compiled with torch.compile (mode={self.compile_mode})")
            except Exception as e:
                print(f"torch.compile failed: {e}")
                self._compiled = False
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        # --- GradNorm logging ---
        grad_norm = None
        # Compute grad norm after backward
        def log_grad_norm(module, grad_input, grad_output):
            total_norm = 0.0
            for p in self.parameters():
                if p.grad is not None:
                    param_norm = p.grad.detach().data.norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm ** 0.5
            nonlocal grad_norm
            if p.grad is not None:
                param_norm = p.grad.detach().data.norm(2)
                grad_norm += param_norm.item() ** 2
        grad_norm = grad_norm ** 0.5
        self.log("grad_norm", grad_norm, prog_bar=True)
        # Log to wandb directly if logger is set
        if self.logger is not None and hasattr(self.logger, "experiment"):
            self.logger.experiment.log({"grad_norm": grad_norm, "epoch": self.current_epoch, "step": self.global_step})

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        # Save misclassified samples for logging
        misclassified = preds != y
        self.log("val_loss", loss, prog_bar=True, sync_dist=True)
        self.log("val_acc", acc, prog_bar=True, sync_dist=True)
        # Collect for confusion matrix
        self.validation_step_outputs.append({"preds": preds.detach().cpu(), "targets": y.detach().cpu()})
        # Return misclassified samples for aggregation
        return {
            "misclassified_images": x[misclassified].detach().cpu(),
            "misclassified_preds": preds[misclassified].detach().cpu(),
            "misclassified_labels": y[misclassified].detach().cpu()
        }

    def on_validation_epoch_end(self):
        # --- Confusion matrix logging ---
        import numpy as np
        import wandb
        import torch
        # Gather all preds and targets
        all_preds = []
        all_targets = []
        for out in self.validation_step_outputs:
            all_preds.append(out["preds"])
            all_targets.append(out["targets"])
        if all_preds and all_targets:
            all_preds = torch.cat(all_preds).numpy()
            all_targets = torch.cat(all_targets).numpy()
            # Compute confusion matrix
            from sklearn.metrics import confusion_matrix
            cm = confusion_matrix(all_targets, all_preds, labels=np.arange(10))
            # Log to wandb
            if self.logger is not None and hasattr(self.logger, "experiment"):
                self.logger.experiment.log({
                    "confusion_matrix": wandb.plot.confusion_matrix(
                        probs=None,
                        y_true=all_targets,
                        preds=all_preds,
                        class_names=[str(i) for i in range(10)]
                    ),
                    "epoch": self.current_epoch
                })
        # Clear for next epoch
        self.validation_step_outputs.clear()
        # --- Existing misclassified logging ---
        # Aggregate misclassified samples from all batches
        misclassified_images = []
        misclassified_preds = []
        misclassified_labels = []
        for out in self.trainer.callback_metrics.get("validation_step_outputs", []):
            if out is not None:
                misclassified_images.append(out["misclassified_images"])
                misclassified_preds.append(out["misclassified_preds"])
                misclassified_labels.append(out["misclassified_labels"])
        if misclassified_images:
            import wandb
            import torch
            images = torch.cat(misclassified_images)
            preds = torch.cat(misclassified_preds)
            labels = torch.cat(misclassified_labels)
            # Limit to 16 samples for logging
            n_samples = min(16, images.size(0))
            images = images[:n_samples]
            preds = preds[:n_samples]
            labels = labels[:n_samples]
            # Prepare wandb.Image objects
            img_list = []
            for i in range(n_samples):
                img = images[i].squeeze().numpy()
                caption = f"pred: {preds[i].item()}, label: {labels[i].item()}"
                img_list.append(wandb.Image(img, caption=caption))
            if self.logger and hasattr(self.logger, "experiment"):
                self.logger.experiment.log({"bad_classifications": img_list, "epoch": self.current_epoch})

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2, verbose=True
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1
            }
        }

In [ ]:
from PIL import ExifTags, Image
Image.ExifTags = ExifTags  # Hack to bypass broken import

In [ ]:
#!pip install matplotlib

In [ ]:
"""
from pytorch_lightning.tuner import Tuner

# 1. Instantiate model and datamodule
model = LitCNN()
dm = MNISTDataModule()



# And you're all set!

# 2. Create trainer WITHOUT auto_lr_find
trainer = pl.Trainer(
    max_epochs=1  # Just for LR finding
)

# Now you can instantiate the Tuner using the Trainer
tuner = Tuner(trainer)
# 3. Run LR finder manually
lr_finder = tuner.lr_find(model, datamodule=dm)

# 4. Plot the result (optional)
fig = lr_finder.plot(suggest=True)
fig.show()

# 5. Update model's learning rate
new_lr = lr_finder.suggestion()
print(f"Suggested LR: {new_lr}")
#model.hparams.lr = new_lr  # assuming your model uses hparams.lr
"""

In [ ]:
import time
from pytorch_lightning.callbacks import Timer
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import LearningRateMonitor

# Train the model
dm = MNISTDataModule()
model = LitCNN()

timer = Timer()

class EpochTimeLogger(pl.Callback):
    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_start_time = time.time()

    def on_train_epoch_end(self, trainer, pl_module):
        epoch_time = time.time() - self.epoch_start_time
        # Log to wandb if logger is set
        if trainer.logger is not None and hasattr(trainer.logger, "experiment"):
            trainer.logger.experiment.log({"epoch_time_sec": epoch_time, "epoch": trainer.current_epoch})

# Add this callback to your trainer
epoch_time_logger = EpochTimeLogger()

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,         # 5-10 is typical for MNIST
    min_delta=1e-4,     # Only stop if improvement is less than this
    mode="min",
    verbose=True,
    strict=True
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    save_top_k=1,
    mode="min",
    save_last=True,
    dirpath="checkpoints",
    filename="mnist-cnn-{epoch:02d}-{val_loss:.2f}"
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

wandb_logger = WandbLogger(project="mnist-cnn-pl")

# 5. Stochastic Weight Averaging (SWA)
# Improve generalization by averaging weights.
from pytorch_lightning.callbacks import StochasticWeightAveraging

swa_callback = StochasticWeightAveraging(swa_lrs=1e-2)

In [ ]:
import torch
num_gpus = torch.cuda.device_count()
trainer = pl.Trainer(
    devices=num_gpus,
    accelerator="auto",
    strategy="auto",  # or remove this line
    
    benchmark=True,
    #strategy="ddp_notebook",
    max_epochs=100, 
    #accelerator="auto", 
    logger=wandb_logger,
    callbacks=[timer, epoch_time_logger, early_stopping, checkpoint_callback, lr_monitor, swa_callback],
    precision="16-mixed"
)
trainer.fit(model, datamodule=dm)

In [ ]:
single_gpu_trainer = pl.Trainer(
    devices=1,
    accelerator="auto"
)
single_gpu_trainer.test(model, datamodule=dm)

## Enable torch.compile (optional)

Set `use_compile=True` when creating the model to enable PyTorch 2.x compilation for faster training (requires PyTorch 2.x+).

In [ ]:
# Example: enable torch.compile (PyTorch 2.x+)
# model = LitCNN(use_compile=True, compile_mode="default")